# 04 Distillation with a control
Goal: measure how much of the ensemble's gain a small student keeps, against a control. The teacher is the average of the tuned LightGBM and a random forest. The code that runs all seeds will live in `src/distill.py`.

## 1. Prediction (written before running anything)

**What is measured.** Each student is trained three ways: on true labels (the control, arm 1), on the teacher's out-of-fold labels for the same rows (arm 2), and on teacher labels for the real rows plus synthetic rows (arm 3). For a student and arm:

> share of the teacher's gain kept = (control RMSE − student RMSE) / (control RMSE − teacher RMSE)

0% means no better than the control, 100% means as good as the teacher, and a negative share means worse than the control.

**Expected shares** (test RMSE, mean over 5 seeds):

| Student | Arm 2: teacher labels, same rows | Arm 3: teacher labels + synthetic rows |
|---|---|---|
| MLP, 2 layers of 32 | about 10% | about 40% |
| MLP, 2 layers of 128 | about 15% | about 50% |
| Depth-limited tree | about 15% | about 35% |

**Why.**
- **Arm 2 helps only a little.** The student sees the same inputs as the control. The teacher's labels are smoother than the true ones, which removes some noise, but nothing new is learned about where the data lies.
- **Arm 3 helps most.** The synthetic rows give the student many more places to copy the teacher's function, and a function is easier to copy than noisy labels. This is where I expect the real gain.
- **The bigger MLP keeps more** because it has the capacity to use the extra rows. The small MLP runs out of capacity first.
- **The tree has the most to gain but the lowest ceiling.** Its control is the weakest, so the teacher's gain over it is large. But a depth-limited tree can only draw a blocky copy of the teacher, so it cannot keep most of that gain.

**What would surprise me.**
- Arm 2 beating arm 3.
- A negative share for any arm, meaning distillation hurt.
- The tree keeping more than either MLP.

A null result still counts. If distillation does not beat the control by more than the seed-to-seed noise, that is what gets reported.

## 2. Build the teacher
The teacher is the average of the tuned LightGBM (from `lightgbm_vs_mlp`) and a random forest of 300 trees, using the `max_features` that gave the best forest in `error_correlation`. `AverageEnsemble` in `src/lib.py` averages them, and can be saved with `joblib`. It is fit on the full training portion, and the test set is used here only to check it.

Note: this notebook trains LightGBM, so it must not load torch. That means no `set_seed` here (it imports torch). Every model below takes its seed directly.

In [1]:
import json

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor

from src.lib import (
    RESULTS_DIR,
    AverageEnsemble,
    EarlyStoppingLGBM,
    evaluate_regression,
    load_data,
    load_split,
)
from src.utils import load_config

cfg = load_config()
seed = cfg["seed"]
X, y = load_data()
X, y = X.to_numpy(), y.to_numpy()
test_idx, folds = load_split()
dev_idx = np.sort(np.concatenate([val for _, val in folds]))  # the full training portion
X_dev, y_dev, X_test, y_test = X[dev_idx], y[dev_idx], X[test_idx], y[test_idx]

In [2]:
# Tuned LightGBM settings, and the best forest setting from the error-correlation sweep.
lgbm_params = json.loads((RESULTS_DIR / "best_lgbm.json").read_text())["params"]
sweep = pd.read_csv(RESULTS_DIR / "error_correlation_sweep.csv")
best_forest = sweep[sweep["bootstrap"]].sort_values("forest_rmse").iloc[0]
rf_max_features = int(best_forest["max_features"])
print("LightGBM:", lgbm_params)
print("random forest: 300 trees, max_features =", rf_max_features)

lgbm = EarlyStoppingLGBM(lgbm_params, seed=seed)
forest = RandomForestRegressor(
    n_estimators=300, max_features=rf_max_features, random_state=seed, n_jobs=-1
)
teacher = AverageEnsemble([lgbm, forest])

LightGBM: {'learning_rate': 0.053400495878730164, 'num_leaves': 104, 'n_estimators': 214, 'min_child_samples': 34, 'subsample': 0.7183586946161812, 'colsample_bytree': 0.6072923364097647, 'reg_lambda': 0.043064020633992874}
random forest: 300 trees, max_features = 4


In [3]:
models = {"LightGBM": lgbm, "Random forest": forest, "Teacher (average)": teacher}
rows = []
for name, model in models.items():
    model.fit(X_dev, y_dev)
    rows.append({"model": name, **evaluate_regression(y_test, model.predict(X_test))})

teacher_table = pd.DataFrame(rows).set_index("model")
teacher_table.round(4)

,rmse,mae,r2
model,,,
LightGBM,0.4413,0.2896,0.8514
Random forest,0.4941,0.3205,0.8137
Teacher (average),0.4578,0.2964,0.8400


In [4]:
# Why the average does not beat LightGBM: how similar are the two models' mistakes?
e_lgbm = lgbm.predict(X_test) - y_test
e_forest = forest.predict(X_test) - y_test
print("error correlation, LightGBM vs. forest:", round(np.corrcoef(e_lgbm, e_forest)[0, 1], 3))

error correlation, LightGBM vs. forest: 0.916


**Takeaways**
- **The teacher beats the random forest but not LightGBM.** Test RMSE is 0.458 for the teacher, 0.494 for the forest and 0.441 for LightGBM. The plan allows this if the reason is stated.
- **Why:** the two models make nearly the same mistakes (error correlation 0.92), so averaging removes very little error. This is the same result as the error-correlation experiment: high correlation means a small gain. And because the forest is the weaker model, the average lands between the two instead of below both.
- **Open decision:** the teacher stays as designed, an equal average, so the control-to-teacher gain in the next steps is measured against 0.458. Using LightGBM alone, or a weighted average chosen by CV on the training rows (not the test set), would give a stronger teacher.